# Step 5
### 3 Experiments Setup

This notebook allows you to easily switch between the three experiments in order to perform fine-tuning:
1. **`head_only`**: Fully frozen ViT encoder. Trains only the classification/mask heads and queries.
2. **`blocks_8_11`**: Unfreezes the last 4 blocks of the ViT + the heads.
3. **`lora`**: Applies Low-Rank Adaptation to the ViT attention layers + trains the heads.

**Instructions:**
Change the `EXP_TYPE` variable in the configuration block to run the desired experiment. The script automatically handles directories, WandB logging, and resuming from checkpoints if training is interrupted.

In [ ]:
import subprocess, sys, os

# 1. Install required dependencies
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "pyyaml", "lightning", "torchmetrics",
    "huggingface_hub", "torchvision", "gitignore-parser", "jsonargparse[signatures]", "wandb", "peft"
], check=True)

import wandb
# NOTE: Replace with your own W&B API key before running. Do not share publicly.
wandb.login(key="YOUR_WANDB_API_KEY")

# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import torch
import torch.nn as nn
import glob
import re
import sys
import os

# Workaround for PyTorch 2.6: torch.load defaults to weights_only=True, which
# breaks Lightning checkpoints containing custom classes. Patch it globally.
if not hasattr(torch, '_is_patched'):
    _orig_load = torch.load
    def safe_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _orig_load(*args, **kwargs)
    torch.load = safe_load
    torch._is_patched = True

# --- Project directory setup ---
PROJECT_ROOT = "/content/drive/My Drive/Comprehensive Road Scene Understanding for Autonomous Driving"
EOMT_DIR   = f"{PROJECT_ROOT}/MaskArchitectureAnomaly_CourseProject/eomt"
DATA_PATH  = f"{PROJECT_ROOT}/datasets/cityscapes"
MODELS_DIR = f"{PROJECT_ROOT}/MaskArchitectureAnomaly_CourseProject/trained_models"

CS_CKPT   = f"{MODELS_DIR}/eomt_cityscapes.bin"
COCO_CKPT = f"{MODELS_DIR}/eomt_coco.bin"

if EOMT_DIR not in sys.path:
    sys.path.insert(0, EOMT_DIR)

print("Directory check:")
for p in [CS_CKPT, COCO_CKPT, DATA_PATH, EOMT_DIR]:
    print("✅ OK     " if os.path.exists(p) else "❌ MISSING", "—", p)

# Remove stale .pyc cache files to avoid import issues after code changes
for f in glob.glob(f"{EOMT_DIR}/training/__pycache__/*.pyc"):
    os.remove(f)

### Configuration: Select your Experiment Here ⌗
Set `EXP_TYPE` to one of the following strings:
- `"head_only"`
- `"blocks_8_11"`
- `"lora"`

In [ ]:
# ── EXPERIMENT CONFIGURATION ─────────────────────────────────────────────────
EXP_TYPE   = "lora"   # Choose one of: "head_only", "blocks_8_11", "lora"
MAX_EPOCHS = 8
LOG_NAME   = f"exp_{EXP_TYPE}"
CKPT_DIR   = f"{EOMT_DIR}/logs/{LOG_NAME}/version_0/checkpoints"

os.environ["WANDB_MODE"] = "online"
print(f"\n Configured for Experiment: {EXP_TYPE.upper()}")

In [ ]:
import lightning as L
from training.mask_classification_semantic import MaskClassificationSemantic
from datasets.cityscapes_semantic import CityscapesSemantic
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger
from models.eomt import EoMT
from models.vit import ViT

def apply_experiment_freezing(model, exp_type):
    """Applies the correct freezing/unfreezing logic based on EXP_TYPE."""
    # Step 1. Freeze all parameters — we will selectively unfreeze below
    for param in model.parameters():
        param.requires_grad = False

    # Step 2. Always unfreeze the prediction heads and query tokens across all experiments
    def is_head_component(name):
        return any(x in name for x in ["class_head", "mask_head", "upscale", "network.q"])

    for name, param in model.named_parameters():
        if is_head_component(name):
            param.requires_grad = True

    # Step 3. Apply experiment-specific unfreezing strategy
    if exp_type == "head_only":
        print("[*] EXP 1: Training Head Only. Encoder fully frozen.")
        # Encoder remains fully frozen; only heads and query tokens are trained

    elif exp_type == "blocks_8_11":
        print("[*] EXP 2: Training Head + Blocks 8-11. Blocks 0-7 are frozen.")
        for name, param in model.named_parameters():
            # Identify DINOv2 ViT backbone blocks by index
            if "encoder.backbone.blocks." in name:
                try:
                    block_idx = int(name.split("blocks.")[1].split(".")[0])
                    if block_idx >= 8:
                        param.requires_grad = True
                except (ValueError, IndexError):
                    pass

    elif exp_type == "lora":
        print("[*] EXP 3: Applying LoRA to the Encoder attention layers.")
        from peft import LoraConfig, get_peft_model

        lora_config = LoraConfig(
            r=8,
            lora_alpha=16,
            target_modules=["qkv"],  # Apply LoRA to QKV attention projections in all ViT blocks
            lora_dropout=0.05,
            bias="none"
        )
        # Wrap the encoder backbone with LoRA adapters via HuggingFace PEFT
        model.network.encoder = get_peft_model(model.network.encoder, lora_config)

        # PEFT wrapping resets requires_grad for all parameters.
        # Re-enable gradients for the prediction heads and query tokens.
        for name, param in model.named_parameters():
            if is_head_component(name):
                param.requires_grad = True

    else:
        raise ValueError(f"Unknown EXP_TYPE: {exp_type}")

    # Report trainable vs. total parameter count for verification
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"\n=> Trainable Params: {trainable / 1e6:.2f} M out of {total / 1e6:.2f} M\n")
    return model

In [ ]:
# ── AUTOMATIC RESUME FROM CHECKPOINT ────────────────────────────────────────
ckpt_files = glob.glob(f"{CKPT_DIR}/*.ckpt")
if ckpt_files:
    def _get_step(path):
        m = re.search(r'step=(\d+)', os.path.basename(path))
        return int(m.group(1)) if m else -1
    last_ckpt = max(ckpt_files, key=_get_step)
    LOAD_COCO  = False   # Resume: weights are restored from checkpoint, skip COCO init
    print(f"[Resume] Resuming from: {last_ckpt}")
else:
    last_ckpt  = None
    LOAD_COCO  = True    # Fresh start: initialize from COCO pretrained weights
    print("[Fresh start] No checkpoint found.")

# ── Network architecture ─────────────────────────────────────────────────────
encoder = ViT(backbone_name="vit_base_patch14_reg4_dinov2", img_size=(640, 640))
network = EoMT(encoder=encoder, num_blocks=3, num_q=200, num_classes=19)

# ── Base model instantiation ─────────────────────────────────────────────────
model = MaskClassificationSemantic(
    network=network,
    img_size=(640, 640),
    num_classes=19,
    attn_mask_annealing_enabled=True,
    attn_mask_annealing_start_steps=[3317, 8292, 13268],
    attn_mask_annealing_end_steps=[6634, 11609, 16585],
    lr=1e-4,
    llrd=0.8,
    llrd_l2_enabled=True,
    lr_mult=1.0,
    weight_decay=0.05,
    poly_power=0.9,
    warmup_steps=[500, 1000],
    ckpt_path=COCO_CKPT if LOAD_COCO else None,  # Load COCO weights on fresh start; None on resume
    load_ckpt_class_head=False,
)

# ── Apply experiment freezing/unfreezing strategy ───────────────────────────
model = apply_experiment_freezing(model, EXP_TYPE)

# ── Datamodule (Cityscapes semantic segmentation) ────────────────────────────
datamodule = CityscapesSemantic(
    path=DATA_PATH,
    batch_size=1,
    img_size=(640, 640),
)

print("Model and datamodule ready ✅")

# ── Trainer configuration ────────────────────────────────────────────────────
checkpoint_cb = ModelCheckpoint(
    every_n_epochs=1,
    save_top_k=-1,
    dirpath=CKPT_DIR,
    filename="epoch={epoch}-step={step}",
)

logger = WandbLogger(
    project="eomt",
    name=LOG_NAME,
    id=f"run_cityscapes_v1_{EXP_TYPE}",
    resume="allow",   # Resume W&B run if the same run ID already exists
)

trainer = L.Trainer(
    max_epochs=MAX_EPOCHS,
    logger=logger,
    callbacks=[checkpoint_cb],
    precision="16-mixed",  # Automatic Mixed Precision (AMP): reduces memory and speeds up training
)

trainer.fit(
    model=model,
    datamodule=datamodule,
    ckpt_path=last_ckpt,
)